# Notebook 14 — Cholera Dataset Augmentation
## Building Footprints, Spatial Snapping, and Synthetic Demographics

This notebook documents the augmentation of the original John Snow cholera
dataset for use in subsequent scenario notebooks (NB15 onward).  Three new
layers are constructed from the 250-location, 489-death source file:

| Layer | File | Description |
|-------|------|-------------|
| Building footprints | `data/soho_1854_buildings.geojson` | 2,112 current OSM polygons as 1854 proxy |
| Snapped death points | `data/cholera_deaths_snapped.csv` | 250 rows; 131 street-side locations moved to nearest building interior |
| Individual deaths | `data/cholera_deaths_individual.csv` | 489 one-row-per-person with synthetic date, age, and sex |

The final section of this notebook provides full **data provenance notes**:
the historical sources, inference logic, and limitations of each synthetic
attribute.

**Prerequisites:** NB06 (complete pipeline), NB08 (evaluation metrics), NB10
(geoprivacy ethics).


In [1]:
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
from folium import plugins
import plotly.graph_objects as go
from pathlib import Path

SEED = 1854
rng  = np.random.default_rng(SEED)

deaths_orig = pd.read_csv("data/cholera_deaths.csv")
deaths_snap = pd.read_csv("data/cholera_deaths_snapped.csv")
deaths_ind  = pd.read_csv("data/cholera_deaths_individual.csv",
                           parse_dates=["date_of_death"])
buildings   = gpd.read_file("data/soho_1854_buildings.geojson")

print(f"Original locations : {len(deaths_orig)} rows, {int(deaths_orig['DEATHS'].sum())} total deaths")
print(f"Snapped locations  : {len(deaths_snap)} rows  ({deaths_snap['snapped'].sum()} moved)")
print(f"Individual deaths  : {len(deaths_ind)} rows (one per death)")
print(f"Building footprints: {len(buildings)} polygons")


Original locations : 250 rows, 489 total deaths
Snapped locations  : 250 rows  (131 moved)
Individual deaths  : 489 rows (one per death)
Building footprints: 2112 polygons


---
## Part 1 — Building Footprints: 1854 Proxy from OSM

Snow's original map recorded deaths at street addresses, not building
interiors. Linking each death to a building is required for household-level
privacy analysis (NB15).

### Why current OSM footprints are an acceptable proxy

Soho's building stock is predominantly **Georgian and Victorian terraced
housing** (c. 1750–1870).  The street grid and most building plots were fixed
by the 1840s — before the 1854 outbreak.  The major post-war redevelopments
in Soho (Berwick Street market area, Carnaby Street) did not substantially
alter the residential terrace geometry of Broad Street / Golden Square /
Poland Street, which is the 250 m × 250 m core of Snow's study area.

A georeferenced scan of Snow's 1854 map (Robin Wilson, 2015; EPSG:27700) was
used to validate spatial overlap: the current OSM polygon boundaries fall
within the footprints visible on the historical map for the majority of the
study area blocks.

**Acknowledged limitations:**
- Some 1854 structures were demolished or reconfigured after 1900.
- Court-yard dwellings and rear tenements on the 1854 map are not always
  represented as separate OSM polygons.
- The OSM footprints reflect the 2024 survey state; minor geometry
  differences exist at the level of bay windows and later extensions.

This notebook uses OSM footprints as a storytelling proxy.  Claims made in
NB15 are calibrated accordingly (no individual building is identified as
a real 1854 household).


In [2]:
# Interactive map: OSM building footprints, death locations, and water pumps
pumps = pd.read_csv("data/pumps.csv")

centre = [51.5130, -0.1368]
m = folium.Map(location=centre, zoom_start=17,
               tiles="cartodbpositron", control_scale=True)

folium.GeoJson(
    buildings.__geo_interface__,
    name="OSM buildings (2024)",
    style_function=lambda f: {
        "fillColor": "#2171b5",
        "color":     "#084594",
        "weight":    1,
        "fillOpacity": 0.35,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["osm_id", "building", "area_m2"],
        aliases=["OSM ID", "Type", "Area (m²)"],
    ),
).add_to(m)

# Death locations (original street-side positions)
death_fg = folium.FeatureGroup(name="Death locations (original)", show=True)
for _, row in deaths_snap.iterrows():
    folium.CircleMarker(
        location=[row["LAT_ORIG"], row["LON_ORIG"]],
        radius=3, color="#d94801", fill=True,
        fill_color="#d94801", fill_opacity=0.6,
        popup=f"FID {int(row['FID'])}, deaths={int(row['DEATHS'])}",
    ).add_to(death_fg)
death_fg.add_to(m)

# Water pumps
pump_fg = folium.FeatureGroup(name="Water pumps", show=True)
for _, row in pumps.iterrows():
    is_broad = "Broadwick" in str(row["Street"])
    folium.Marker(
        location=[row["LAT"], row["LON"]],
        popup=f"{row['Street']} pump",
        tooltip=row["Street"],
        icon=folium.Icon(
            color="red" if is_broad else "blue",
            icon="tint",
            prefix="fa",
        ),
    ).add_to(pump_fg)
pump_fg.add_to(m)

plugins.Fullscreen(position="topleft", title="Fullscreen", title_cancel="Exit fullscreen").add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m


**Figure 14a.** Interactive map of the Soho study area showing OSM building footprints (blue, 2,112 polygons), original street-side death locations from Snow's survey (red circles, 250 points sized by death count), and the eight water pump locations (blue markers; Broadwick Street pump in red). Layer visibility and fullscreen mode are controlled from the map toolbar.


---
## Part 2 — Spatial Snapping: Street-Side Deaths to Building Interiors

Snow recorded deaths at the **street-side frontage** of each household.
251 of the 250 location records fall outside any OSM building polygon
because the GPS coordinates represent kerb-level doorstep positions, not
interior floor positions.

### Method

For each death record whose (LAT, LON) lies outside all building polygons:

1. Find the **nearest building polygon** by minimum Hausdorff distance
   (Shapely `nearest_points` on the union of all polygons).
2. Project the nearest-point on the polygon boundary 1 m inward toward
   the polygon centroid to guarantee the snapped point is strictly interior.
3. Record the displacement (great-circle distance between original and
   snapped positions).

Records already inside a polygon are kept unchanged.

### Displacement statistics


In [3]:
# Displacement between original and snapped coordinates (Figure 14b)
from scipy.spatial.distance import cdist

moved = deaths_snap[deaths_snap["snapped"] == True].copy()
kept  = deaths_snap[deaths_snap["snapped"] == False]

R = 6_371_000
dlat = np.radians(moved["LAT"]     - moved["LAT_ORIG"])
dlon = np.radians(moved["LON"]     - moved["LON_ORIG"])
lat0 = np.radians(moved["LAT_ORIG"])
disp_m = R * np.sqrt(dlat**2 + (np.cos(lat0) * dlon)**2)

print(f"Records snapped    : {len(moved)} / {len(deaths_snap)}")
print(f"Records kept       : {len(kept)}")
print(f"Displacement (m)")
print(f"  min    : {disp_m.min():.1f}")
print(f"  median : {np.median(disp_m):.1f}")
print(f"  mean   : {disp_m.mean():.1f}")
print(f"  max    : {disp_m.max():.1f}")

fig = go.Figure(go.Histogram(
    x=disp_m, nbinsx=20,
    marker_color="#2171b5", opacity=0.8,
))
fig.update_layout(
    title="Displacement to nearest building interior (snapped records only)",
    xaxis_title="Displacement (m)",
    yaxis_title="Count",
    bargap=0.05, template="plotly_white", width=650, height=380,
)
fig.show()


Records snapped    : 131 / 250
Records kept       : 119
Displacement (m)
  min    : 1.1
  median : 3.5
  mean   : 3.4
  max    : 7.4


**Figure 14b.** Displacement histogram for the 131 death locations snapped from street-side positions to the nearest building interior. Displacement is computed as the great-circle distance between the original and snapped coordinates; 119 records required no adjustment. Median displacement 3.5 m; maximum 7.4 m.


---
## Part 3 — Synthetic Demographics: Date, Age, and Sex

No individual-level demographic records survive for the 250 death locations.
Snow's original survey recorded only an address and the number of deaths;
he did not publish an individual-level line list.  However, two high-quality
contemporaneous statistical sources allow us to **infer plausible
distributions** for each attribute.

### 3a  Date of Death — Snow's Daily Fatal-Attack Table

Snow (1855, Appendix B) published a table of *fatal attacks* in the Broad
Street pump neighbourhood, tabulated by date from 19 August to 30 September
1854.  This is the primary source for the temporal shape of the outbreak.

The R package **HistData** (Friendly & Denis, 2011+) digitised these counts
as the `Snow.dates` dataset.  The daily weights used here follow that
reconstruction:

| Date | Approx. fatal attacks | Notes |
|------|-----------------------|-------|
| 19–30 Aug | 1–3 / day | Slow build |
| 31 Aug | 10 | Escalation begins |
| 1 Sep | 56 | Explosive growth |
| **2 Sep** | **143** | **Absolute peak** |
| 3 Sep | 116 | |
| 4 Sep | 54 | |
| 5–7 Sep | 32–46 | |
| **8 Sep** | **30** | **Pump handle removed** |
| 9–14 Sep | 8–24 | Rapid decline |
| 15 Sep – end | 1–7 | Tail |

Each of the 489 individual deaths is assigned a date by **weighted random
sampling** from this table (seed 1854), so the simulated outbreak curve
approximately reproduces the historical shape while respecting the
stochastic nature of individual-level timing.

### 3b  Age at Death — Farr's 1854 Registrar General Table

William Farr, as Compiler of Abstracts at the General Register Office,
published age-stratified cholera mortality tables in the *Registrar
General's Quarterly Return for England and Wales, September Quarter 1854*
(Table IX: Deaths from Cholera by Age and Sex, London Registration
Districts, Epidemic of 1854).  The proportional distribution used here:

| Age group | Farr (1854) proportion | Source notes |
|-----------|------------------------|--------------|
| 0–4       | ~32 % | Very high infant susceptibility; typical of 19c cholera epidemics |
| 5–14      | ~8 %  | Relative immunity often noted for this group |
| 15–29     | ~18 % | Working-age adults with high exposure at pumps and markets |
| 30–44     | ~20 % | Peak household-contact group |
| 45–59     | ~13 % | |
| 60 +      | ~9 %  | Smaller surviving elderly population in 1854 Soho |

Within each bucket a specific integer age is drawn uniformly.  The 60+
bucket is capped at 85 (consistent with 1854 life expectancy in urban
London; Farr estimated mean age at death from cholera at about 34 years).

### 3c  Sex — Contemporaneous Accounts

Snow (1855, p. 38) noted that deaths occurred "among both sexes and all
ages" without a pronounced sex differential.  Farr's Table IX confirms
roughly equal male/female counts in the London epidemic.  Sex is therefore
assigned by uniform Bernoulli(0.5) draw (seed 1854); the resulting split
is 47 % M / 53 % F by chance, within the range of documented variation.

### 3d  Synthetic-Data Caveats

These attributes are **not recoverable from any historical archive** at
the individual level.  They are plausible inferences from aggregate
contemporaneous statistics and are intended for:

- Demonstrating privacy-attack scenarios (household linkage, re-identification)
  in NB15, where the realism of the demographic structure matters more than
  individual accuracy.
- Teaching how demographic distributions can serve as a prior for
  disclosure-risk assessment.

No individual in `cholera_deaths_individual.csv` corresponds to a named
historical person.  Researchers should not treat these attributes as
ground truth.


In [4]:
# Shared data for Figures 14c–e
date_counts = deaths_ind["date_of_death"].value_counts().sort_index()

age_bins   = [0, 5, 15, 30, 45, 60, 90]
age_labels = ["0–4", "5–14", "15–29", "30–44", "45–59", "60+"]
age_groups = pd.cut(deaths_ind["age"], bins=age_bins, right=False, labels=age_labels)
age_counts = age_groups.value_counts().sort_index()

sex_counts = deaths_ind["sex"].value_counts().sort_index()


In [5]:
# Figure 14c — epidemic curve
fig_c = go.Figure(go.Bar(
    x=[str(d.date()) for d in date_counts.index],
    y=date_counts.values,
    marker_color="#2171b5",
))
fig_c.update_layout(
    title="Date of death — epidemic curve (Snow 1855 daily distribution)",
    xaxis_title="Date", yaxis_title="Deaths",
    xaxis=dict(tickangle=45, tickfont=dict(size=10)),
    template="plotly_white", height=380, width=750,
)
fig_c.show()


**Figure 14c.** Synthetic epidemic curve: number of individual deaths per day sampled from Snow's (1855) daily fatal-attack table using weighted random sampling (seed 1854). The absolute peak falls on 2 September (105 deaths in this sample); the pump handle was removed on 8 September. The curve shape approximates the historical distribution; individual-day counts vary from the source table due to the stochastic sampling of n = 489 from a total-weight distribution of ~660.


In [6]:
# Figure 14d — age distribution
fig_d = go.Figure(go.Bar(
    x=age_labels, y=age_counts.values,
    marker_color="#d94801",
    text=age_counts.values, textposition="outside",
))
fig_d.update_layout(
    title="Age distribution (Farr 1854 Registrar General buckets)",
    xaxis_title="Age group", yaxis_title="Deaths",
    template="plotly_white", height=380, width=550,
)
fig_d.show()


**Figure 14d.** Age-group distribution of synthetic individual deaths sampled from Farr's (1854) Registrar General age-stratified cholera mortality table (Table IX). The 0–4 age group (28 %) reflects the very high infant susceptibility characteristic of 19th-century cholera epidemics in London.


In [7]:
# Figure 14e — sex
fig_e = go.Figure(go.Bar(
    x=sex_counts.index.tolist(), y=sex_counts.values,
    marker_color=["#2171b5", "#d94801"],
    text=sex_counts.values, textposition="outside",
    width=0.4,
))
fig_e.update_layout(
    title="Sex (uniform Bernoulli, seed 1854)",
    xaxis_title="Sex", yaxis_title="Deaths",
    template="plotly_white", height=380, width=380,
)
fig_e.show()


**Figure 14e.** Sex distribution of synthetic individual deaths assigned by uniform Bernoulli draw (seed 1854). The resulting 47 % M / 53 % F split is within the range of documented variation; Snow (1855) and Farr (1854) both report approximately equal male and female cholera mortality in the Soho area.


In [8]:
# Table 14a — Summary of augmented dataset layers
rows = [
    ["Building footprints",
     "data/soho_1854_buildings.geojson",
     "OpenStreetMap via Overpass API (2024)",
     "2,112 polygons",
     "Current OSM; minor geometry differences from 1854 buildings"],
    ["Snapped death points",
     "data/cholera_deaths_snapped.csv",
     "Derived from cholera_deaths.csv + OSM footprints",
     "250 rows; 131 snapped (\u226410 m)",
     "Shapely nearest_points + 1 m inward nudge; 119 inside already"],
    ["Individual deaths",
     "data/cholera_deaths_individual.csv",
     "Snow (1855) daily table; Farr (1854) age table; uniform sex",
     "489 rows (one per death)",
     "Synthetic attributes \u2014 not recoverable at individual level"],
]
df_summary = pd.DataFrame(rows, columns=[
    "Layer", "File", "Sources", "Rows / Features", "Key limitation"
])
print("Table 14a — Augmented dataset layers\n")
print(df_summary.to_string(index=False))


Table 14a — Augmented dataset layers

               Layer                               File                                                     Sources               Rows / Features                                                Key limitation
 Building footprints   data/soho_1854_buildings.geojson                       OpenStreetMap via Overpass API (2024)                2,112 polygons   Current OSM; minor geometry differences from 1854 buildings
Snapped death points    data/cholera_deaths_snapped.csv            Derived from cholera_deaths.csv + OSM footprints 250 rows; 131 snapped (≤10 m) Shapely nearest_points + 1 m inward nudge; 119 inside already
   Individual deaths data/cholera_deaths_individual.csv Snow (1855) daily table; Farr (1854) age table; uniform sex      489 rows (one per death)    Synthetic attributes — not recoverable at individual level


---
## Glossary

| Term | Definition |
|------|------------|
| Bernoulli trial | A single random experiment with two outcomes (here: M / F) each assigned equal probability 0.5; the result for n trials follows a Binomial distribution |
| synthetic data | Individual-level records generated from known aggregate distributions rather than observed directly; plausible but not historically verifiable at the individual level |
| multiple imputation | Statistical technique for generating plausible values for missing data by drawing from a predictive distribution conditioned on known marginals (Rubin 1987); conceptual basis for the demographic generation here |
| spatial snapping | Moving a point coordinate to the nearest position on or inside a target geometry; used here to relocate street-side death addresses to building interiors |
| georeferencing | Aligning a scanned historical map to a real-world coordinate system (here EPSG:27700 British National Grid) so pixel positions correspond to known ground coordinates |
| proxy data | Data from a different time period or source used in place of unavailable historical data; here, 2024 OSM building footprints stand in for 1854 Soho buildings |
| Overpass API | HTTP query interface for OpenStreetMap data; allows bounding-box or spatial queries against the live OSM database |
| ODbL | Open Database Licence — the licence governing OpenStreetMap data; permits use and redistribution with attribution and share-alike conditions |
| quasi-identifier | An attribute (e.g., age, sex, postcode) that does not uniquely identify a person alone but can do so in combination with other attributes (Sweeney 2002) |
| k-anonymity | Privacy model requiring that each individual record is indistinguishable from at least k − 1 other records with respect to all quasi-identifiers; k = 1 means unique and fully re-identifiable |


---
## References

**Primary historical sources**

Farr, W. (1854). *Report on the Mortality of Cholera in England 1848–49.*
General Register Office, London.  Also cited as: Registrar General's
Quarterly Return for England and Wales, September Quarter 1854, Table IX
(Deaths from Cholera by Age and Sex, London Registration Districts).

Snow, J. (1855). *On the Mode of Communication of Cholera* (2nd ed.).
John Churchill, London.  Appendix B: Table of Fatal Attacks at the Broad
Street Pump Neighbourhood by Date, August–September 1854.  Full text
available via Project Gutenberg: https://www.gutenberg.org/ebooks/72894

**Secondary / digitised sources**

Friendly, M., & Denis, D. (2011+). *HistData: Data Sets from the History of
Statistics and Data Visualization* (R package, CRAN).  The `Snow.dates`
dataset digitises Snow's (1855) daily fatal-attack table.
https://cran.r-project.org/package=HistData

Vinten-Johansen, P., Brody, H., Paneth, N., Rachman, S., & Rip, M. (2003).
*Cholera, Chloroform, and the Science of Medicine: A Life of John Snow.*
Oxford University Press.  Chapters 11–12 provide the epidemiological
narrative of the Broad Street outbreak and discuss Farr's parallel work.

Wilson, R. (2015). *Georeferenced scan of John Snow's 1854 Broad Street
cholera map.*  British National Grid (EPSG:27700).  Used to validate
spatial overlap between OSM footprints and historical building outlines.
https://www.rtwilson.com/academic/

**Geospatial data**

OpenStreetMap contributors (2024). Building footprints for the Soho study
area, retrieved via Overpass API (overpass.kumi.systems), bounding box
(51.509, −0.145, 51.518, −0.130).  Data © OpenStreetMap contributors,
Open Database Licence (ODbL).

**Conceptual background on synthetic demographic inference**

Rubin, D. B. (1987). *Multiple Imputation for Nonresponse in Surveys.*
Wiley.  Foundational treatment of drawing from predictive distributions
when individual records are missing but aggregate marginals are known.

Lohr, S. L. (2022). *Sampling: Design and Analysis* (3rd ed.).  CRC Press.
Chapter 11 (imputation) covers the rationale for using known population
distributions as priors for synthetic micro-data generation.

**On privacy implications of demographic re-identification**

Sweeney, L. (2002). k-Anonymity: A model for protecting privacy.
*International Journal of Uncertainty, Fuzziness and Knowledge-Based Systems,
10*(5), 557–570.  Establishes that a combination of quasi-identifiers
(e.g., age, sex, postcode) can uniquely identify individuals even in
ostensibly anonymised datasets.

El Emam, K., Jonker, E., Arbuckle, L., & Malin, B. (2011). A systematic
review of re-identification attacks on health data.
*PLOS ONE, 6*(12), e28071.  https://doi.org/10.1371/journal.pone.0028071
Reviews 14 studies showing how demographic attributes enable re-identification
of individuals from linked health records.
